# Bangla ASR test: Vosk Bengali **streaming Zipformer** (sherpa-onnx)

`alphacep/vosk-model-small-streaming-bn`: small Zipformer2 transducer trained with k2/icefall,
**streaming / real-time**, ~94 MB, **Apache-2.0**. Card WER: Common Voice 17.9, Fleurs 20.6,
Kathbath 19.3, Kathbath-noisy 22.5, SUBAK.KO 24.6.

The notebook **simulates a live call**: audio is fed in 100 ms pieces, and built-in endpoint
detection splits speech into timestamped utterances on pauses.

> **RTF** = processing time / audio duration. Below 1.0 is faster than real time.
> Use **Runtime > Change runtime type > CPU** to measure CPU-server speed.

## 1. Settings

In [ ]:
DECODING = "modified_beam_search"   # or "greedy_search" (faster)
BEAM = 4
THREADS = 4               # threads per stream (a live server runs many streams in parallel)
PIECE_SEC = 0.1           # audio fed per step (simulated live stream)
PAUSE_SEC = 0.8           # silence after speech that ends an utterance
ENDPOINT = True           # split into utterances

AUDIO_SOURCE = "upload"   # "upload" or "drive"
DRIVE_AUDIO_DIR = "/content/drive/MyDrive/sample-audio"

## 2. Install dependencies (~1 minute)

In [ ]:
!apt-get -qq install -y ffmpeg libsndfile1 > /dev/null
!pip install -q sherpa-onnx huggingface_hub librosa soundfile jiwer

## 3. Imports and environment check

In [ ]:
import os, sys, time, json, shutil, traceback
from pathlib import Path
import numpy as np, torch, librosa
if THREADS:
    torch.set_num_threads(THREADS)
print("python :", sys.version.split()[0])
print("torch  :", torch.__version__)
print("cores  :", os.cpu_count(), "| torch threads:", torch.get_num_threads())
!lscpu | grep -E "Model name|^CPU\(s\)"
import sherpa_onnx
print("sherpa-onnx:", sherpa_onnx.__version__)

## 4. Download the model (~94 MB)

In [ ]:
os.environ["HF_HUB_DISABLE_XET"] = "1"   # Xet transfer can stall; plain HTTP is reliable
from huggingface_hub import snapshot_download
REPO_ID = "alphacep/vosk-model-small-streaming-bn"
MODEL_DIR = Path("/content/model")
token = None

t0 = time.time()
snapshot_download(REPO_ID, local_dir=MODEL_DIR, token=token, allow_patterns=['am-onnx/*.onnx', 'lang/*', 'test.wav', 'README.md'])
print(f"Downloaded in {time.time()-t0:.0f}s")
!du -sh /content/model; ls -R /content/model | head -30

## 5. Streaming helper

In [ ]:
SR = 16000

def build_recognizer():
    am, lang = MODEL_DIR / "am-onnx", MODEL_DIR / "lang"
    kw = dict(encoder=str(am / "encoder.onnx"), decoder=str(am / "decoder.onnx"),
              joiner=str(am / "joiner.onnx"), tokens=str(lang / "tokens.txt"),
              num_threads=THREADS, sample_rate=SR, feature_dim=80,
              decoding_method=DECODING, max_active_paths=BEAM, provider="cpu",
              model_type="zipformer2", enable_endpoint_detection=ENDPOINT,
              rule1_min_trailing_silence=2.4, rule2_min_trailing_silence=PAUSE_SEC,
              rule3_min_utterance_length=20.0)
    try:
        return sherpa_onnx.OnlineRecognizer.from_transducer(dither=3e-5, **kw)
    except TypeError:
        return sherpa_onnx.OnlineRecognizer.from_transducer(**kw)

def result_text(rec, s):
    r = rec.get_result(s)
    return (r if isinstance(r, str) else getattr(r, "text", str(r))).strip()

def stream_file(rec, audio):
    s = rec.create_stream()
    piece = max(1, int(PIECE_SEC * SR))
    segments, seg_start, fed = [], 0.0, 0
    def flush(end_t):
        nonlocal seg_start
        t = result_text(rec, s)
        if t:
            segments.append((round(seg_start, 2), round(end_t, 2), t))
        rec.reset(s)
        seg_start = end_t
    for i in range(0, len(audio), piece):
        s.accept_waveform(SR, audio[i:i + piece])
        fed += len(audio[i:i + piece])
        while rec.is_ready(s):
            rec.decode_stream(s)
        if rec.is_endpoint(s):
            flush(fed / SR)
    s.accept_waveform(SR, np.zeros(int(0.66 * SR), dtype=np.float32))
    s.input_finished()
    while rec.is_ready(s):
        rec.decode_stream(s)
    flush(len(audio) / SR)
    return segments

## 6. Load the model

In [ ]:
t0 = time.time()
recognizer = build_recognizer()
LOAD_SEC = time.time() - t0
print(f"Model loaded in {LOAD_SEC:.1f}s")

## 7. Get your audio
* **upload**: select audio files, plus optional reference transcripts named like the audio
  (`call.mp3` + `call.txt`) to get WER/CER.
* **drive**: mounts Google Drive and reads `DRIVE_AUDIO_DIR`.

In [ ]:
AUDIO_DIR = Path("/content/sample-audio")
AUDIO_DIR.mkdir(exist_ok=True)
AUDIO_EXTS = {".wav", ".mp3", ".flac", ".ogg", ".m4a", ".opus", ".webm"}

if AUDIO_SOURCE == "upload":
    from google.colab import files
    for name, data in files.upload().items():
        (AUDIO_DIR / name).write_bytes(data)
elif AUDIO_SOURCE == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    for p in Path(DRIVE_AUDIO_DIR).iterdir():
        if p.suffix.lower() in AUDIO_EXTS | {".txt"}:
            shutil.copy(p, AUDIO_DIR / p.name)

AUDIO_FILES = sorted(p for p in AUDIO_DIR.iterdir() if p.suffix.lower() in AUDIO_EXTS)
print(f"{len(AUDIO_FILES)} audio file(s):")
for p in AUDIO_FILES:
    print(" -", p.name + (" (+ reference .txt)" if p.with_suffix(".txt").exists() else ""))

if (MODEL_DIR / "test.wav").exists():   # sample clip shipped with the model
    shutil.copy(MODEL_DIR / "test.wav", AUDIO_DIR / "vosk_test.wav")
    AUDIO_FILES = sorted(p for p in AUDIO_DIR.iterdir() if p.suffix.lower() in AUDIO_EXTS)

## 8. Run the test (simulated live stream)

In [ ]:
import jiwer
results, errors = [], []
for f in AUDIO_FILES:
    print("=" * 80)
    print("FILE:", f.name)
    rec = {"file": f.name, "status": "ok"}
    try:
        audio, _ = librosa.load(str(f), sr=SR, mono=True)
        if audio.size == 0:
            raise ValueError("audio decoded to 0 samples")
        dur = len(audio) / SR
        t0 = time.time()
        segs = stream_file(recognizer, audio)
        infer = time.time() - t0
        text = " ".join(t for _, _, t in segs)
        rec.update(duration_sec=round(dur, 2), inference_sec=round(infer, 2), rtf=round(infer / dur, 3),
                   segments_n=len(segs), text=text,
                   segments=[{"start": a, "end": b, "text": t} for a, b, t in segs])
        print(f"duration {dur:.2f}s | inference {infer:.2f}s | RTF {infer/dur:.3f} | segments {len(segs)}")
        for a, b, t in segs:
            print(f"  [{a:7.2f} - {b:7.2f}] {t}")
        ref_file = f.with_suffix(".txt")
        if ref_file.exists():
            ref = ref_file.read_text(encoding="utf-8").strip()
            rec.update(wer=round(jiwer.wer(ref, text), 4), cer=round(jiwer.cer(ref, text), 4))
            print(f"WER {rec['wer']:.2%} | CER {rec['cer']:.2%}")
    except Exception as e:
        rec.update(status="error", error=f"{type(e).__name__}: {e}")
        errors.append((f.name, traceback.format_exc()))
        print("ERROR:", rec["error"])
    results.append(rec)

## 9. Summary

In [ ]:
import pandas as pd
pd.set_option("display.max_colwidth", 120)
df = pd.DataFrame(results)
display(df[[c for c in df.columns if c not in ("text", "segments")]])
ok = df[df.status == "ok"]
if len(ok):
    print(f"Model load {LOAD_SEC:.1f}s | total audio {ok.duration_sec.sum():.1f}s | "
          f"inference {ok.inference_sec.sum():.1f}s | overall RTF {ok.inference_sec.sum()/ok.duration_sec.sum():.3f}")
print(f"{len(ok)} ok, {len(errors)} failed")

## 10. Errors (full tracebacks)

In [ ]:
if not errors:
    print("No errors")
for name, tb in errors:
    print("=" * 80)
    print("FILE:", name)
    print(tb)

## 11. Save and download results

In [ ]:
from datetime import datetime
out_json = f"/content/sherpa_zipformer_bn_streaming_results_{datetime.now():%Y%m%d_%H%M%S}.json"
with open(out_json, "w", encoding="utf-8") as fh:
    json.dump({"model": REPO_ID, "decoding": DECODING, "threads": torch.get_num_threads(), "load_sec": round(LOAD_SEC, 2),
               "results": results, "errors": [{"file": n, "traceback": t} for n, t in errors]},
              fh, ensure_ascii=False, indent=2)
print("Saved", out_json)
from google.colab import files
files.download(out_json)